# Prompt Creation with Prompt Studios

## Learning Objectives

By completing this notebook, you will:
1. Create effective prompts using Dataiku Prompt Studios
2. Use template variables for dynamic content
3. Implement few-shot learning with examples
4. Design prompts for structured output
5. Apply prompt engineering best practices

## Prerequisites

- Module 0 completed (LLM Mesh setup)
- Understanding of LLM capabilities and limitations
- Familiarity with your domain (commodity markets, finance, etc.)

## Estimated Time: 50 minutes

---

In [ ]:
learning_objectives(["Create effective prompts using Dataiku Prompt Studios", "Use template variables for dynamic content", "Implement few-shot learning with examples", "Design prompts for structured output", "Apply prompt engineering best practices"])

## Why Prompt Engineering Matters

**The Problem**: LLMs are powerful but sensitive to how you ask. Poor prompts lead to:
- Inconsistent outputs
- Hallucinations
- Wrong format
- Missing key information

**The Solution**: Systematic prompt design with:
- Clear instructions
- Relevant context
- Examples (few-shot learning)
- Output format specification

### Key Insight

**Prompt quality matters more than model selection**. A well-engineered prompt with GPT-3.5 often outperforms a poorly-written prompt with GPT-4.

### Dataiku Prompt Studios

Prompt Studios provides a visual interface for:
1. **Designing prompts** with system and user messages
2. **Testing with variables** using template syntax
3. **Versioning prompts** to track improvements
4. **Sharing prompts** across projects and teams

## Setup

Import libraries for working with prompts programmatically.

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import dataiku
from dataiku.llm import LLM
from dataiku.prompt import PromptStudio
import json
from typing import Dict, List
import re

## Exercise 1: Basic Prompt Structure

A good prompt has three key components:

1. **Role/Context**: Who is the AI and what's the situation?
2. **Task**: What should it do?
3. **Constraints**: Output format, length, what to avoid

### Anatomy of a Good Prompt

```
[ROLE] You are an expert commodity analyst.

[CONTEXT] You analyze daily market reports to extract trading signals.

[TASK] Analyze the following report and determine if it's bullish or bearish for crude oil.

[FORMAT] Respond in JSON with fields: sentiment (bullish/bearish/neutral), confidence (high/medium/low), reason (one sentence).

[INPUT] {{report_text}}
```

In [ ]:
# Example: Bad prompt vs Good prompt

llm = LLM('claude-sonnet')  # Update with your connection

test_report = """EIA reported crude oil inventories fell by 6.3 million barrels, 
exceeding analyst expectations of a 2.5 million barrel draw. Refinery utilization 
increased to 91.2% from 89.8% the previous week."""

# Bad prompt - vague, no structure
bad_prompt = f"What does this mean? {test_report}"

response_bad = llm.complete(bad_prompt, max_tokens=100)
print("=== BAD PROMPT RESPONSE ===")
print(response_bad.text)
print()

# Good prompt - structured, clear expectations
good_prompt = f"""You are a commodity market analyst specializing in oil markets.

Task: Analyze this EIA inventory report and determine the market sentiment.

Report: {test_report}

Provide your analysis in JSON format with these exact fields:
- sentiment: "bullish", "bearish", or "neutral"
- confidence: "high", "medium", or "low"
- key_factor: one sentence explaining the main driver

JSON:"""

response_good = llm.complete(good_prompt, max_tokens=150, temperature=0)
print("=== GOOD PROMPT RESPONSE ===")
print(response_good.text)

# Parse and validate
try:
    data = json.loads(response_good.text)
    print("\n✓ Valid JSON with expected structure")
    print(f"Sentiment: {data.get('sentiment')}")
except:
    print("\n✗ Failed to parse as JSON")

### Exercise 1.1: Write a Structured Prompt

**Task**: Create a prompt to extract key data points from commodity price reports.

Your prompt should:
- Define the AI's role as a data extraction specialist
- Specify exactly what fields to extract
- Request JSON output
- Handle missing data gracefully

In [ ]:
def create_extraction_prompt(report_text: str) -> str:
    """
    Create a prompt to extract commodity data.
    
    Should extract:
    - commodity: name of the commodity
    - current_price: latest price with unit
    - price_change: change amount and direction
    - change_percent: percentage change
    - key_driver: main reason for price movement
    
    Returns JSON string.
    """
    # YOUR CODE HERE
    prompt = f"""
    
    """
    
    return prompt

# Test your prompt
test_cases = [
    """WTI crude futures settled at $82.45 per barrel, up $3.12 or 3.9% on the day. 
    The rally was driven by OPEC+ production cuts announced this morning.""",
    
    """Natural gas prices plunged to $2.34/MMBtu, down 8.2% as unseasonably warm 
    weather reduced heating demand across the Northeast.""",
    
    """Gold held steady near $2,050 per ounce with minimal trading activity ahead 
    of tomorrow's Federal Reserve decision."""
]

for i, test_case in enumerate(test_cases, 1):
    print(f"\n=== Test Case {i} ===")
    prompt = create_extraction_prompt(test_case)
    response = llm.complete(prompt, max_tokens=200, temperature=0)
    
    try:
        data = json.loads(response.text)
        print("✓ Valid JSON")
        print(f"Commodity: {data.get('commodity')}")
        print(f"Price: {data.get('current_price')}")
        print(f"Change: {data.get('price_change')} ({data.get('change_percent')})")
        print(f"Driver: {data.get('key_driver')}")
        
        # Auto-graded checks
        assert 'commodity' in data, "Missing 'commodity' field"
        assert 'current_price' in data, "Missing 'current_price' field"
        assert data['commodity'] is not None, "Commodity should not be null"
        
    except json.JSONDecodeError:
        print(f"✗ Invalid JSON: {response.text[:100]}")
        raise

print("\n✓ Exercise 1.1 passed!")

## Exercise 2: Template Variables

Dataiku Prompt Studios supports template variables using `{{variable_name}}` syntax. This allows you to:
- Reuse prompts with different inputs
- Test prompts systematically
- Deploy prompts as reusable assets

### Template Syntax

```
Simple substitution:
  {{variable_name}}

Conditional:
  {{#if condition}}
    content if true
  {{/if}}

Loop:
  {{#each items}}
    Process {{this}}
  {{/each}}
```

In [ ]:
# Simulating Prompt Studio template system

class PromptTemplate:
    """Simple template engine for prompts."""
    
    def __init__(self, template: str):
        self.template = template
    
    def render(self, **variables) -> str:
        """Replace {{variable}} with values."""
        result = self.template
        for key, value in variables.items():
            result = result.replace(f"{{{{{key}}}}}", str(value))
        return result

# Example: Sentiment analysis template
sentiment_template = PromptTemplate("""
You are a {{domain}} market analyst.

Analyze this {{report_type}} report for market sentiment.

Report: {{report_text}}

Respond in JSON format:
{
  "sentiment": "bullish" | "bearish" | "neutral",
  "confidence": "high" | "medium" | "low",
  "summary": "one sentence summary"
}

JSON:
""")

# Test with different variables
test_scenarios = [
    {
        'domain': 'crude oil',
        'report_type': 'EIA inventory',
        'report_text': 'Crude stocks fell 5.1M barrels, gasoline stocks rose 2.3M barrels.'
    },
    {
        'domain': 'agricultural commodities',
        'report_type': 'USDA crop',
        'report_text': 'Corn yields revised up 3% due to favorable weather in August.'
    }
]

for scenario in test_scenarios:
    print(f"\n=== {scenario['domain'].upper()} ===")
    prompt = sentiment_template.render(**scenario)
    response = llm.complete(prompt, max_tokens=150, temperature=0)
    
    data = json.loads(response.text)
    print(f"Sentiment: {data['sentiment']} (confidence: {data['confidence']})")
    print(f"Summary: {data['summary']}")

### Exercise 2.1: Create a Multi-Purpose Template

**Task**: Design a template that can be used for multiple commodity types and report formats.

Template variables should include:
- `commodity`: Name of commodity (e.g., "crude oil", "natural gas")
- `analysis_type`: Type of analysis ("sentiment", "forecast", "summary")
- `report_text`: The actual report content
- `output_format`: Desired format ("json", "bullet_points", "paragraph")

In [ ]:
def create_flexible_template() -> PromptTemplate:
    """
    Create a template that adapts to different use cases.
    """
    # YOUR CODE HERE
    template_text = """
    
    """
    
    return PromptTemplate(template_text)

# Test cases
template = create_flexible_template()

test_scenarios = [
    {
        'commodity': 'WTI crude oil',
        'analysis_type': 'sentiment',
        'report_text': 'Inventory draw of 4.2M barrels exceeded expectations.',
        'output_format': 'json'
    },
    {
        'commodity': 'natural gas',
        'analysis_type': 'forecast',
        'report_text': 'Storage levels 5% below 5-year average entering winter.',
        'output_format': 'bullet_points'
    },
    {
        'commodity': 'copper',
        'analysis_type': 'summary',
        'report_text': 'LME copper inventories down 15% month-over-month. Chinese demand showing signs of recovery.',
        'output_format': 'paragraph'
    }
]

for i, scenario in enumerate(test_scenarios, 1):
    print(f"\n=== Scenario {i}: {scenario['commodity']} - {scenario['analysis_type']} ===")
    prompt = template.render(**scenario)
    response = llm.complete(prompt, max_tokens=200, temperature=0.3)
    print(response.text[:300] + "..." if len(response.text) > 300 else response.text)
    
    # Auto-graded checks
    assert scenario['commodity'].lower() in prompt.lower(), "Template should include commodity"
    assert scenario['analysis_type'].lower() in prompt.lower(), "Template should mention analysis type"

print("\n✓ Exercise 2.1 passed!")

## Exercise 3: Few-Shot Learning

**Few-shot learning** means providing examples in your prompt. This dramatically improves:
- Output consistency
- Format adherence
- Domain-specific behavior

### Pattern

```
[Instructions]

Examples:

Input: [example 1 input]
Output: [example 1 output]

Input: [example 2 input]
Output: [example 2 output]

Now process this:

Input: {{user_input}}
Output:
```

In [ ]:
# Example: Classification without few-shot
zero_shot_prompt = """Classify this report as: INVENTORY_REPORT, PRODUCTION_NEWS, PRICE_MOVEMENT, or OTHER.

Report: EIA reported commercial crude oil inventories increased by 1.2 million barrels.

Classification:"""

response = llm.complete(zero_shot_prompt, max_tokens=20, temperature=0)
print("Zero-shot result:", response.text.strip())

# With few-shot examples
few_shot_prompt = """Classify commodity reports into categories: INVENTORY_REPORT, PRODUCTION_NEWS, PRICE_MOVEMENT, or OTHER.

Examples:

Report: "U.S. crude oil production reached 13.2 million barrels per day, a new record."
Classification: PRODUCTION_NEWS

Report: "WTI crude settled up $2.45 at $78.32 per barrel."
Classification: PRICE_MOVEMENT

Report: "EIA reported commercial crude oil inventories increased by 1.2 million barrels."
Classification: INVENTORY_REPORT

Report: "OPEC Secretary General discussed global oil market outlook at conference."
Classification: OTHER

Now classify:

Report: "Gasoline stocks rose by 3.4 million barrels while distillate inventories fell."
Classification:"""

response = llm.complete(few_shot_prompt, max_tokens=20, temperature=0)
print("Few-shot result:", response.text.strip())
print("\nNotice: Few-shot is more reliable and consistent!")

### Exercise 3.1: Build a Few-Shot Classifier

**Task**: Create a sentiment classifier using few-shot learning.

Your classifier should:
- Classify as BULLISH, BEARISH, or NEUTRAL
- Include 2-3 examples per category
- Work across different commodities

In [ ]:
def create_few_shot_sentiment_classifier() -> str:
    """
    Create a few-shot prompt for sentiment classification.
    
    Should include:
    - Clear task description
    - 2-3 examples of BULLISH reports
    - 2-3 examples of BEARISH reports  
    - 1-2 examples of NEUTRAL reports
    - Template for new input
    """
    # YOUR CODE HERE
    prompt = """
    
    """
    
    return prompt

# Test the classifier
base_prompt = create_few_shot_sentiment_classifier()

test_reports = [
    "Crude inventories plummeted 7.5M barrels, the largest draw in six months.",
    "Oil prices drifted sideways on mixed signals from OPEC+ and demand concerns.",
    "Supply glut fears intensified as U.S. production hit new highs while demand weakened.",
    "Refinery maintenance season begins with minimal market impact expected."
]

expected_sentiments = ['BULLISH', 'NEUTRAL', 'BEARISH', 'NEUTRAL']

correct = 0
for report, expected in zip(test_reports, expected_sentiments):
    prompt = base_prompt + f"\n\nReport: \"{report}\"\nSentiment:"
    response = llm.complete(prompt, max_tokens=10, temperature=0)
    
    predicted = response.text.strip().upper()
    is_correct = expected in predicted
    correct += int(is_correct)
    
    print(f"Report: {report[:50]}...")
    print(f"Expected: {expected}, Predicted: {predicted} {'✓' if is_correct else '✗'}")
    print()

accuracy = correct / len(test_reports)
print(f"Accuracy: {accuracy:.1%}")

# Auto-graded test
assert accuracy >= 0.5, "Classifier should get at least 50% correct"
assert 'BULLISH' in base_prompt, "Prompt should include BULLISH examples"
assert 'BEARISH' in base_prompt, "Prompt should include BEARISH examples"

print("✓ Exercise 3.1 passed!")

## Exercise 4: Prompt Versioning and Testing

In production, prompts evolve. Dataiku Prompt Studios supports versioning to:
- Track changes over time
- A/B test different versions
- Roll back if needed

We'll simulate this by testing prompt variations systematically.

In [ ]:
# Simulate prompt versions

class PromptVersion:
    """Represents a versioned prompt."""
    
    def __init__(self, version: str, template: str, description: str):
        self.version = version
        self.template = template
        self.description = description
        self.metrics = {'total_tests': 0, 'successes': 0, 'avg_tokens': 0}
    
    def test(self, test_cases: List[dict], llm: LLM) -> dict:
        """Test prompt version on multiple cases."""
        results = []
        total_tokens = 0
        
        for test_case in test_cases:
            prompt = self.template.format(**test_case['input'])
            response = llm.complete(prompt, max_tokens=200, temperature=0)
            
            # Check if output matches expected format
            success = test_case['validator'](response.text)
            
            results.append({
                'input': test_case['input'],
                'output': response.text,
                'success': success,
                'tokens': response.usage.total_tokens
            })
            
            total_tokens += response.usage.total_tokens
        
        # Update metrics
        self.metrics['total_tests'] = len(results)
        self.metrics['successes'] = sum(r['success'] for r in results)
        self.metrics['avg_tokens'] = total_tokens / len(results) if results else 0
        
        return {
            'version': self.version,
            'success_rate': self.metrics['successes'] / self.metrics['total_tests'],
            'avg_tokens': self.metrics['avg_tokens'],
            'results': results
        }

# Define prompt versions
v1 = PromptVersion(
    version='v1.0',
    template='Extract the commodity name from: {text}',
    description='Basic extraction'
)

v2 = PromptVersion(
    version='v2.0',
    template='''Extract the commodity name from this report. 
Return only the commodity name, nothing else.

Report: {text}

Commodity:''',
    description='More explicit instructions'
)

v3 = PromptVersion(
    version='v3.0',
    template='''Extract the commodity name.

Examples:
Report: "WTI crude settled at $80/bbl" → WTI crude oil
Report: "Natural gas prices fell 5%" → Natural gas

Report: {text}
Commodity:''',
    description='With few-shot examples'
)

# Test cases
def validate_commodity_extraction(output: str) -> bool:
    """Check if output is a valid commodity name."""
    output_clean = output.strip().lower()
    valid_commodities = ['crude', 'oil', 'gas', 'gold', 'copper', 'wheat', 'corn']
    return any(c in output_clean for c in valid_commodities) and len(output_clean.split()) <= 5

test_cases = [
    {'input': {'text': 'Brent crude futures rose to $85.20 per barrel.'},
     'validator': validate_commodity_extraction},
    {'input': {'text': 'Gold prices held near record highs.'},
     'validator': validate_commodity_extraction},
    {'input': {'text': 'Henry Hub natural gas dropped 3.2%.'},
     'validator': validate_commodity_extraction},
]

# Test all versions
print("Testing prompt versions...\n")
for version in [v1, v2, v3]:
    result = version.test(test_cases, llm)
    print(f"{version.version}: {version.description}")
    print(f"  Success rate: {result['success_rate']:.1%}")
    print(f"  Avg tokens: {result['avg_tokens']:.0f}")
    print()

### Exercise 4.1: Create and Test Prompt Variants

**Task**: Create 3 versions of a prompt for price change extraction and test which performs best.

Your prompts should extract:
- Direction: up/down
- Amount: numerical change
- Percentage: if mentioned

In [ ]:
# YOUR CODE HERE
# Create 3 PromptVersion instances with different approaches
# Test them on price change extraction

def validate_price_extraction(output: str) -> bool:
    """Validate price change extraction."""
    output_lower = output.lower()
    has_direction = 'up' in output_lower or 'down' in output_lower or 'rose' in output_lower or 'fell' in output_lower
    has_number = any(c.isdigit() for c in output)
    return has_direction and has_number

price_test_cases = [
    {'input': {'text': 'Oil prices jumped $3.45 to close at $82.10.'},
     'validator': validate_price_extraction},
    {'input': {'text': 'Futures fell 4.2% to $65.80 per barrel.'},
     'validator': validate_price_extraction},
    {'input': {'text': 'Crude settled down $1.25 or 1.5% at $78.50.'},
     'validator': validate_price_extraction},
]

# Create your prompt versions
my_v1 = PromptVersion(
    version='my_v1',
    template='',  # YOUR CODE HERE
    description='Your description'
)

my_v2 = PromptVersion(
    version='my_v2', 
    template='',  # YOUR CODE HERE
    description='Your description'
)

my_v3 = PromptVersion(
    version='my_v3',
    template='',  # YOUR CODE HERE
    description='Your description'
)

# Test all versions
results = []
for version in [my_v1, my_v2, my_v3]:
    result = version.test(price_test_cases, llm)
    results.append(result)
    print(f"{result['version']}: {result['success_rate']:.1%} success, {result['avg_tokens']:.0f} avg tokens")

# Find best version
best = max(results, key=lambda r: r['success_rate'])
print(f"\nBest version: {best['version']} ({best['success_rate']:.1%} success rate)")

# Auto-graded test
assert any(r['success_rate'] >= 0.67 for r in results), "At least one version should achieve 67%+ success"
print("\n✓ Exercise 4.1 passed!")

## Summary

Congratulations! You've mastered prompt creation in Dataiku. Key takeaways:

1. **Structure matters** - Role, Task, Format, Examples
2. **Template variables** enable reusable prompts
3. **Few-shot learning** dramatically improves consistency
4. **Version and test** prompts systematically
5. **Dataiku Prompt Studios** provides visual tools for all of this

## Best Practices

- Start simple, add complexity only if needed
- Use examples for new or ambiguous tasks
- Test on diverse inputs, not just happy paths
- Version prompts when making significant changes
- Monitor performance in production

## Next Steps

- **Notebook 02**: Advanced prompt testing and evaluation
- **Module 2**: Combine prompts with RAG for knowledge-grounded generation
- **Module 3**: Deploy prompts in production applications

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])